# 05 — The three graph conditions: safe, leaky, null (issues #6, #7, #8)

This is the core experimental design of the whole project: the same graph-building pipeline (issues #4/#5), run three different ways, so "does a leakage-safe graph help prediction" has something real to be compared against.

- **Safe (issue #6)**: each patient's history is cut off strictly *before* their own `prediction_time` -- the honest, deployable version.
- **Leaky (issue #7)**: each patient's *entire* history, including anything after `prediction_time` -- a deliberate negative control. If safe and leaky end up nearly identical, the censoring logic probably isn't doing anything (a bug worth catching, not a good result).
- **Null (issue #8)**: the safe graph's structure, scrambled -- same number of connections per patient (degree-preserving `double_edge_swap`), same distribution of edge strengths, but *who* connects to *whom* is randomized. Tests whether a graph helps because of real medical similarity, or because any graph-shaped structure would help.

**Node identity matters here**: issue #3 found that `guo_los` patients can have several different `prediction_time` rows (multiple hospital admissions). Each one needs its *own* view of "history so far" -- so the graph's nodes here are `(patient_id, prediction_time)` pairs, not just patients. `ehrgraph.graphs.vectorize.build_node_vectors` (new this issue) handles that, reusing the exact same `events_before`/`events_all` functions from issue #1 for the actual time-censoring -- the safety-critical boundary logic isn't reimplemented, just reused.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ehrgraph.data.loader import open_database
from ehrgraph.graphs.null_model import build_null_graph
from ehrgraph.graphs.similarity import (
    cosine_similarity_matrix,
    graph_summary,
    mutual_knn,
    to_networkx,
    to_tfidf,
    top_k_sparsify,
    union_knn,
)
from ehrgraph.graphs.vectorize import build_node_vectors

plt.rcParams["figure.facecolor"] = "white"

## Build the node list: every `(patient_id, prediction_time)` row for `guo_los`

Not deduplicated by patient -- issue #3's finding was that each row is a separate prediction opportunity with its own valid history cutoff.

In [ ]:
labels = pd.read_csv(
    assets / "benchmark" / "guo_los" / "labeled_patients.csv", parse_dates=["prediction_time"]
)
db = open_database(str(meds_reader_path))
db_ids = set(db)
labels = labels[labels["patient_id"].isin(db_ids)]
nodes = list(zip(labels["patient_id"], labels["prediction_time"], strict=True))
n_patients = labels["patient_id"].nunique()
print(f"{len(nodes)} nodes ((patient_id, prediction_time) pairs), {n_patients} distinct patients")

## Build the safe and leaky vectors -- same code, one flag flipped

`censor=True` calls `events_before` (issue #6); `censor=False` calls `events_all` (issue #7). Everything else about the vectorization is identical.

In [ ]:
safe_matrix, safe_nodes, safe_codes = build_node_vectors(
    db, nodes, table="condition", censor=True, weighting="binary", dedup="per_patient"
)
leaky_matrix, leaky_nodes, leaky_codes = build_node_vectors(
    db, nodes, table="condition", censor=False, weighting="binary", dedup="per_patient"
)
print(f"safe:  {safe_matrix.shape}, {safe_matrix.nnz:,} nonzero, {len(safe_codes):,} codes")
print(f"leaky: {leaky_matrix.shape}, {leaky_matrix.nnz:,} nonzero, {len(leaky_codes):,} codes")
ratio = leaky_matrix.nnz / safe_matrix.nnz
print(
    f"leaky sees {ratio:.2f}x as many (node, code) entries as safe -- future history is real "
    "information, and the safe condition is deliberately not allowed to use it."
)

## Build both graphs through the same similarity pipeline (issue #5)

In [ ]:
safe_sim = cosine_similarity_matrix(to_tfidf(safe_matrix))
leaky_sim = cosine_similarity_matrix(to_tfidf(leaky_matrix))


def to_graph(sim_matrix, node_order, k=10, symmetrize=mutual_knn):
    top_k = top_k_sparsify(sim_matrix, k=k)
    return to_networkx(symmetrize(top_k), node_order)


G_safe = to_graph(safe_sim, safe_nodes)
G_leaky = to_graph(leaky_sim, leaky_nodes)

summary_df = pd.DataFrame(
    [
        {"condition": "safe (issue #6)", **graph_summary(G_safe)},
        {"condition": "leaky (issue #7)", **graph_summary(G_leaky)},
    ]
).set_index("condition")
summary_df

In [ ]:
safe_edges = {frozenset(e) for e in G_safe.edges()}
leaky_edges = {frozenset(e) for e in G_leaky.edges()}
print(f"edge sets differ: {safe_edges != leaky_edges}")
print(f"edges only in safe:  {len(safe_edges - leaky_edges):,}")
print(f"edges only in leaky: {len(leaky_edges - safe_edges):,}")
print(f"edges in both:       {len(safe_edges & leaky_edges):,}")
assert safe_edges != leaky_edges, (
    "censoring did nothing -- stop and investigate (issue #9's exact failure mode)"
)

### A real, counter-intuitive finding -- worth checking, not just reporting

Naively you'd expect the leaky graph -- strictly more information per patient -- to look "better" (denser, more connected). It doesn't: more isolated nodes, a smaller largest connected component. Before writing that down as the finding, it's worth asking *why*, and checking the answer against the actual numbers rather than guessing -- a wrong-but-plausible-sounding explanation is worse than no explanation.

### Checking the mechanism directly, not assuming it

First guess: maybe leaky's much larger vectors (109 codes/node on average vs. 51) just *dilute* similarity -- more codes, mostly not shared between any given pair, so cosine similarity drops. That guess turns out to be **wrong**, checked below.

In [ ]:
codes_per_node_safe = np.asarray(safe_matrix.sum(axis=1)).flatten()
codes_per_node_leaky = np.asarray(leaky_matrix.sum(axis=1)).flatten()
print(
    f"mean codes/node -- safe: {codes_per_node_safe.mean():.1f}  "
    f"leaky: {codes_per_node_leaky.mean():.1f}"
)

safe_density = safe_sim.nnz / (safe_sim.shape[0] * safe_sim.shape[1])
leaky_density = leaky_sim.nnz / (leaky_sim.shape[0] * leaky_sim.shape[1])
print(
    f"mean nonzero cosine similarity -- safe: {safe_sim.data.mean():.4f}  "
    f"leaky: {leaky_sim.data.mean():.4f}"
)
print(
    f"median nonzero cosine similarity -- safe: {np.median(safe_sim.data):.4f}  "
    f"leaky: {np.median(leaky_sim.data):.4f}"
)
print(
    f"similarity-matrix density (any nonzero pair) -- safe: {safe_density:.1%}  "
    f"leaky: {leaky_density:.1%}"
)

**The dilution guess is wrong.** Leaky's average and median *nonzero* similarity are not lower than safe's -- they're marginally *higher*. And leaky patients are similar to a much *larger* fraction of the cohort before any pruning (71% vs. 45% of all pairs have some nonzero similarity). Leakage really is adding real signal, exactly as it should -- the raw similarity computation isn't the source of the fragmentation.

**What actually explains it: mutual k-NN's reciprocity requirement.** `mutual_knn` only keeps an edge if *both* patients pick each other among their top 10. Leaky's much larger, noisier vectors (lifetime history vs. a snapshot up to one moment) spread similarity more evenly across many candidates -- more near-ties, less separation between a patient's 10th-best match and their 50th-best. That makes it *less* likely any two specific patients land in each other's exact top 10, even with more raw similarity available overall. Safe's shorter, more concentrated, currently-relevant vectors produce sharper rankings, so mutual agreement is easier to reach. This is testable directly: if the mechanism is right, the fragmentation gap should shrink under `union_knn` (either side picking the other is enough -- no reciprocity required).

### Robustness check: does the gap shrink under union k-NN, as the mechanism predicts?

In [ ]:
G_safe_union = to_graph(safe_sim, safe_nodes, symmetrize=union_knn)
G_leaky_union = to_graph(leaky_sim, leaky_nodes, symmetrize=union_knn)

sparsify_check_df = pd.DataFrame(
    [
        {"condition": "safe, mutual", **graph_summary(G_safe)},
        {"condition": "leaky, mutual", **graph_summary(G_leaky)},
        {"condition": "safe, union", **graph_summary(G_safe_union)},
        {"condition": "leaky, union", **graph_summary(G_leaky_union)},
    ]
).set_index("condition")
sparsify_check_df[["isolated_fraction", "largest_cc_fraction"]]

Confirms the mechanism: under `union_knn`, the safe-vs-leaky isolated-node gap shrinks sharply (and may close or reverse -- read the printed numbers above for this run, don't assume the values from an earlier run still hold). That's consistent with the fragmentation being a `mutual_knn`-reciprocity artifact specific to how much larger and flatter leaky's similarity landscape is, not evidence that leakage adds less real signal -- the raw-similarity diagnostic above already ruled that out directly.

**What this does and doesn't tell us.** It doesn't tell us anything about whether leakage inflates *predictive* performance -- that's a separate, not-yet-run comparison (issues #15/#17, later, actually training models on these graphs). A more fragmented leaky graph under mutual k-NN could still leak plenty of outcome-relevant signal through the specific neighbors it keeps. This structural check only establishes that the safe/leaky distinction is real, substantial, and mechanistically understood -- not a silent no-op, and not a sign of a broken pipeline.

## Build the null graph (issue #8) from the safe graph

Degree-preserving edge swaps + the real graph's own weight multiset reassigned at random -- not `nx.configuration_model`, which would silently discard the weights.

In [ ]:
G_null = build_null_graph(G_safe, seed=0)

real_degrees = sorted(d for _, d in G_safe.degree())
null_degrees = sorted(d for _, d in G_null.degree())
real_weights = sorted(w for _, _, w in G_safe.edges(data="weight"))
null_weights = sorted(w for _, _, w in G_null.edges(data="weight"))

print("degree sequence preserved exactly:", real_degrees == null_degrees)
print("weight distribution preserved exactly:", real_weights == null_weights)

null_edges = {frozenset(e) for e in G_null.edges()}
n_shared = len(safe_edges & null_edges)
print(f"edges shared with the real safe graph: {n_shared:,} of {len(safe_edges):,}")

In [ ]:
three_way_df = pd.DataFrame(
    [
        {"condition": "safe (issue #6)", **graph_summary(G_safe)},
        {"condition": "leaky (issue #7)", **graph_summary(G_leaky)},
        {"condition": "null (issue #8)", **graph_summary(G_null)},
    ]
).set_index("condition")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
three_way_df[["isolated_fraction"]].plot(kind="bar", ax=axes[0], legend=False, color="#C44E52")
axes[0].set_title("Fraction of nodes with zero connections")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

three_way_df[["largest_cc_fraction"]].plot(kind="bar", ax=axes[1], legend=False, color="#4C72B0")
axes[1].set_title("Fraction of nodes in the largest connected cluster")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()
three_way_df

**Reading the three-way comparison:** the null graph is actually the *most* connected of the three (highest largest-CC fraction) -- makes sense, since randomly reshuffling edges among nodes with the same degrees tends to spread connectivity out more evenly than real medical similarity does (real similarity clusters people into somewhat separate, tighter groups rather than one big blob). That's a legitimate, useful sanity check on its own: if the null graph were LESS connected than the real graphs, something would likely be wrong with the swap procedure.

## Summary -- issues #6, #7, #8 done-when checklist

- **#6 (safe)**: graph exists for Task A (`guo_los`), built with `events_before` (time-censored) via `build_node_vectors(..., censor=True)`. The censoring boundary itself is tested directly in `tests/test_censoring.py` (unchanged from issue #1); this notebook confirms it plugs into the real graph pipeline correctly, on real data.
- **#7 (leaky)**: graph exists for Task A, edge set confirmed different from the safe graph's (asserted above, not eyeballed) -- the exact counts of edges unique to each condition are printed above.
- **#8 (null)**: null graph exists for Task A, same degree sequence as the real (safe) graph (`sorted(degrees) == sorted(degrees)`, confirmed `True` above), weight distribution preserved (confirmed `True` above).
- **Bonus, closed out while this machinery existed to test it**: issue #9's fourth and final test (determinism -- same seed gives an identical graph hash) is now written and passing in `tests/test_censoring.py`, completing that issue's full 4/4 test requirement.
- **The leaky-graph fragmentation finding, checked rather than just reported**: the leaky graph is more fragmented than the safe graph under mutual k-NN. The first hypothesis for why (similarity "dilution" from longer vectors) was checked directly against the data and found to be **wrong** -- leaky's raw similarity is if anything slightly higher, and far more pairs have some nonzero similarity. The actual mechanism is `mutual_knn`'s reciprocity requirement interacting with leaky's flatter, noisier similarity rankings -- confirmed by the fact that the gap shrinks under `union_knn`, which doesn't require reciprocity. This means the finding is well-understood, not a sign of a broken pipeline, and it specifically does **not** speak to whether leakage inflates predictive performance -- that comparison hasn't been run yet (issues #15/#17).